# FastAPI Backend Development — Detailed Notes
## Database Operations, ORM Relationships, Migrations, and Connection Pooling

These notes explain the database layer of a FastAPI application using an online shopping API as a running example. The goal is to understand the purpose of each component, how the pieces fit together, and the trade-offs involved in production systems.

### Application flow

A typical request moves through the following stages:

`Client → FastAPI → Request validation → Dependencies → Business logic → ORM/session → Database → Response schema → Client`

The exact order of dependency resolution, middleware, exception handling, and serialization depends on the application. This diagram is a useful high-level model.

### Topics covered
1. CRUD operations and transaction boundaries
2. SQLAlchemy sessions: `add`, `flush`, `commit`, `refresh`, and `rollback`
3. Request and response schemas
4. One-to-many and many-to-many relationships
5. Cascade behavior and data-retention decisions
6. The N+1 query problem and eager loading
7. Alembic migrations and safe schema evolution
8. PostgreSQL versus SQLite
9. Connection pooling and session lifecycle
10. Transactions, concurrency, and preventing overselling
11. A complete order-creation request flow
12. Interview-ready review questions

# 1. CRUD: The Four Basic Database Operations

CRUD stands for **Create, Read, Update, and Delete**. Most APIs expose some variation of these operations.

| Operation | Common HTTP method | Example |
|---|---|---|
| Create | `POST` | Add a product |
| Read | `GET` | Retrieve a product or list products |
| Update | `PUT` or `PATCH` | Replace a resource or modify selected fields |
| Delete | `DELETE` | Remove or deactivate a product |

HTTP methods describe the requested operation; the application must still validate the request, enforce authorization, and handle database errors.

## Example scenario

An online shop has products with an ID, name, price, and available stock. A request to create a product may follow this path:

1. FastAPI receives the request.
2. The request body is validated against a Pydantic schema.
3. The endpoint or service applies business rules.
4. SQLAlchemy tracks the new ORM object.
5. A transaction writes the change to the database.
6. The application returns a response matching its response schema.

The API should not report success before it knows the operation has succeeded. It should also avoid exposing internal fields such as password hashes or private operational data.

# 2. SQLAlchemy Session Lifecycle

A SQLAlchemy `Session` manages ORM object state and coordinates database transactions. It is not the same thing as a physical database connection.

A session may acquire a connection from the engine's pool when database work is needed. After the session is closed, a checked-out connection is normally returned to the pool for reuse.

## The important methods

| Method | Purpose |
|---|---|
| `session.add(obj)` | Add an object to the session so its changes can be tracked |
| `session.flush()` | Send pending SQL operations to the database within the current transaction |
| `session.commit()` | Flush pending work and commit the current transaction |
| `session.refresh(obj)` | Reload an object's values from the database |
| `session.rollback()` | Roll back the current transaction |
| `session.close()` | End the session's use and release its resources |

### `add()` is not the same as `commit()`

Calling `add()` usually does not immediately insert a row. SQLAlchemy tracks the object. SQL may be sent later when a flush occurs, including automatically before certain queries or commits.

### Why use `flush()`?

A database may generate an ID when a row is inserted. Sometimes that ID is needed to create related records before the whole operation is ready to commit. A flush can issue the insert and make the generated ID available while leaving the transaction uncommitted.

### Why use `commit()`?

Commit marks the transaction as successful and makes its changes durable according to the database's transaction guarantees. If the operation fails before commit, the application can roll back the transaction.

### Why use `refresh()`?

A database can generate or modify values, such as a primary key, server-side default, or trigger-generated field. Refresh reloads the object from the database. Depending on the configuration and operation, generated values may already be available without an explicit refresh.

### Important recovery rule

After a failed flush or transaction error, a SQLAlchemy session often requires an explicit `rollback()` before it can be reused. Do not catch an exception and continue issuing database operations in a failed transaction without recovery.

In [ ]:
# Illustrative SQLAlchemy pattern.
# Product and SessionLocal are assumed to be defined elsewhere.

def create_product(name: str, price: float, db):
    product = Product(name=name, price=price)
    db.add(product)

    try:
        db.commit()
        db.refresh(product)
        return product
    except Exception:
        db.rollback()
        raise

The example illustrates the transaction pattern, but production code should use validated request schemas, appropriate exception handling, logging, and a well-defined session lifecycle. Avoid catching every exception merely to return a generic success-shaped response.

# 3. Request Schemas, ORM Models, and Response Schemas

A database model and an API schema serve different purposes.

- **ORM model:** represents database tables, columns, constraints, and relationships.
- **Request schema:** defines which input fields a client may submit and how they are validated.
- **Response schema:** defines which fields the API returns.

Keeping these responsibilities separate makes the API contract clearer and reduces accidental exposure of internal data.

For example, a database user record may include `password_hash`, `is_admin`, and internal audit fields. A public response should expose only fields that are appropriate for that endpoint.

A response schema is a serialization and output-contract boundary; it is not a substitute for authorization. The application must still check whether the caller may access the requested resource.

In [ ]:
from pydantic import BaseModel, ConfigDict, Field

class ProductCreate(BaseModel):
    name: str = Field(min_length=1, max_length=200)
    price: float = Field(gt=0)

class ProductUpdate(BaseModel):
    # Optional fields allow a partial update.
    name: str | None = Field(default=None, min_length=1, max_length=200)
    price: float | None = Field(default=None, gt=0)

class ProductResponse(BaseModel):
    model_config = ConfigDict(from_attributes=True)

    id: int
    name: str
    price: float

`from_attributes=True` allows Pydantic v2 to build a response model from object attributes, including ORM objects. In a real application, decide deliberately how `None` and omitted fields should behave in update requests. For a PATCH operation, `model_dump(exclude_unset=True)` is commonly used to distinguish omitted fields from fields explicitly supplied by the client.

# 4. CRUD Patterns

## Create

Typical sequence:
1. Validate input.
2. Apply business rules.
3. Construct an ORM object.
4. Add it to the session.
5. Commit the transaction.
6. Return a response that matches the API contract.

## Read

Use a targeted query rather than loading an entire table and filtering in Python. For primary-key lookups, `Session.get(Model, primary_key)` is a convenient option.

## Update

Retrieve the record, confirm it exists, apply validated changes, and commit. For partial updates, update only fields supplied by the client.

## Delete

Confirm the record exists, determine whether deletion is allowed, delete or deactivate it, and commit. Consider foreign-key constraints, historical records, audit requirements, and retention policies before choosing hard deletion.

### PUT versus PATCH

- `PUT` generally represents replacement of a resource's state.
- `PATCH` represents partial modification.

The precise semantics depend on the API contract. A PATCH implementation should distinguish an omitted field from a field explicitly set to `null`.

In [ ]:
# Illustrative CRUD query patterns; assumes Product, select, and db exist.

# Read one record by primary key:
product = db.get(Product, 10)

# Read a filtered, ordered, limited list:
statement = (
    select(Product)
    .where(Product.price < 60000)
    .order_by(Product.price)
    .limit(20)
)
products = db.scalars(statement).all()

# For a partial update, validated values might be applied like this:
# changes = payload.model_dump(exclude_unset=True)
# for field, value in changes.items():
#     setattr(product, field, value)
# db.commit()

# For deletion:
# db.delete(product)
# db.commit()

# 5. Transactions and Atomicity

A transaction groups related database changes into one logical unit. Order creation is a useful example.

An order may require:
- One row in `orders`
- Multiple rows in `order_items`
- Inventory updates for the purchased products

If the order row is created but a later inventory update fails, the database must not be left with a partially completed order. A transaction lets the application commit the related changes together or roll them back.

A simplified pattern is:

```python
try:
    db.add(order)
    db.flush()  # May make the generated order ID available.
    db.add_all(order_items)
    # Apply inventory updates.
    db.commit()
except Exception:
    db.rollback()
    raise
```

This is a conceptual example, not a complete inventory implementation. Correct concurrency handling is also needed when multiple customers can buy the same stock.

## ACID in brief

- **Atomicity:** a transaction's changes succeed as a unit or are rolled back.
- **Consistency:** constraints and application rules keep data valid.
- **Isolation:** concurrent transactions are controlled according to the database's isolation level.
- **Durability:** committed changes persist according to the database's guarantees.

Do not keep transactions open while waiting on unrelated slow external calls. Long transactions can hold locks and connections, increasing contention and pool pressure.

# 6. ORM Relationships

Relationships let the ORM navigate between objects. Foreign keys enforce references at the database level; ORM relationship attributes provide object-level navigation and loading behavior.

## One-to-many: User and Order

One user can have many orders, while each order belongs to one user.

The `orders` table stores a foreign key such as `user_id` referencing `users.id`.

## Many-to-many: Order and Product

An order can contain multiple products, and a product can appear in multiple orders. An association table connects them.

For an order system, the association usually needs additional fields such as `quantity` and `unit_price`, so it is often best represented as an `OrderItem` model rather than a bare association table.

Preserving the purchase-time unit price is important because a product's current price may change after an order has been placed.

In [ ]:
# Illustrative SQLAlchemy 2.x relationship models.
# A shared Base is assumed to be defined elsewhere.

from sqlalchemy import ForeignKey, String, Numeric
from sqlalchemy.orm import Mapped, mapped_column, relationship

class User(Base):
    __tablename__ = "users"

    id: Mapped[int] = mapped_column(primary_key=True)
    name: Mapped[str] = mapped_column(String(100))

    orders: Mapped[list["Order"]] = relationship(
        back_populates="user"
    )

class Order(Base):
    __tablename__ = "orders"

    id: Mapped[int] = mapped_column(primary_key=True)
    user_id: Mapped[int] = mapped_column(ForeignKey("users.id"))

    user: Mapped["User"] = relationship(
        back_populates="orders"
    )
    items: Mapped[list["OrderItem"]] = relationship(
        back_populates="order"
    )

class Product(Base):
    __tablename__ = "products"

    id: Mapped[int] = mapped_column(primary_key=True)
    name: Mapped[str] = mapped_column(String(200))
    price: Mapped[float] = mapped_column(Numeric(10, 2))

    order_items: Mapped[list["OrderItem"]] = relationship(
        back_populates="product"
    )

class OrderItem(Base):
    __tablename__ = "order_items"

    id: Mapped[int] = mapped_column(primary_key=True)
    order_id: Mapped[int] = mapped_column(ForeignKey("orders.id"))
    product_id: Mapped[int] = mapped_column(ForeignKey("products.id"))
    quantity: Mapped[int]
    unit_price: Mapped[float] = mapped_column(Numeric(10, 2))

    order: Mapped["Order"] = relationship(back_populates="items")
    product: Mapped["Product"] = relationship(back_populates="order_items")

The model snippet is illustrative. A complete application should add suitable constraints, indexes, currency and decimal handling, validation, and deletion policies. For financial amounts, use decimal/numeric types rather than binary floating-point arithmetic.

## Cascade behavior

Cascade settings determine how operations on one ORM object affect related ORM objects. Database foreign keys may also define actions such as `ON DELETE CASCADE`, `RESTRICT`, or `SET NULL`. ORM cascades and database-level actions are related but distinct.

Choose deletion behavior based on business requirements:
- Restrict deletion when dependent records must remain valid.
- Use soft deletion when historical records should be retained.
- Cascade deletion only when removing dependent data is genuinely intended.
- Use `SET NULL` only when the foreign-key column and data model allow it.

For example, deleting a user should not automatically erase historical orders if those orders must be retained for accounting or support.

# 7. The N+1 Query Problem

The N+1 problem occurs when an initial query loads a collection and then each object causes another query to load related data.

For example:
- One query loads 100 orders.
- Accessing each order's user may cause up to 100 additional queries.

That can mean as many as 101 queries, depending on relationship loading, identity-map reuse, and the data involved.

## How to reduce it

SQLAlchemy offers eager-loading strategies such as:
- `selectinload()`: loads the parent collection and then loads related rows using a separate query, often with an `IN` clause.
- `joinedload()`: loads related data through a join.

Choose based on relationship cardinality, result size, and the shape of the query. Eagerly loading every relationship can itself waste memory and database work.

In [ ]:
from sqlalchemy import select
from sqlalchemy.orm import selectinload

# Assumes Order and its `user` relationship are defined.
statement = select(Order).options(selectinload(Order.user))
orders = db.scalars(statement).all()

# The related users are generally fetched in a separate query,
# avoiding one query per order for this relationship.

**Interview explanation:** “The N+1 problem is when a collection query is followed by extra queries for each object's related data. I would inspect the SQL and use an appropriate eager-loading strategy, such as `selectinload` or `joinedload`, based on the relationship and expected result size.”

# 8. Alembic Database Migrations

Applications evolve. A `products` table may initially contain `id`, `name`, and `price`; later, a new requirement adds `category`.

Dropping and recreating the production table is unsafe because it can destroy existing data. A migration describes a controlled schema change.

Alembic is commonly used with SQLAlchemy to track and apply versioned database schema changes.

## Typical workflow

1. Change the ORM model.
2. Generate a migration revision.
3. Review the generated migration.
4. Test the migration against a representative database.
5. Apply the migration in the deployment process.
6. Verify the resulting schema.

Common commands:

```bash
alembic revision --autogenerate -m "add product category"
alembic upgrade head
alembic downgrade -1
```

- `revision --autogenerate` proposes migration operations based on differences between configured metadata and the database schema.
- `upgrade head` applies migrations up to the latest revision.
- `downgrade -1` attempts to reverse one revision.

## Why review generated migrations?

Autogeneration is not guaranteed to detect or correctly represent every intended change. Renames, data transformations, constraints, indexes, and database-specific operations may require manual edits.

A downgrade can also lose data. A migration that drops a column cannot necessarily restore the values that were in it.

## Safe production schema evolution

For a new required column in a table that already contains rows, a safer sequence may be:
1. Add the column in a backward-compatible way.
2. Populate existing rows.
3. Add or enforce the required constraint.
4. Deploy code that depends on the new field.

The exact plan depends on table size, database capabilities, deployment order, and whether old and new application versions must run at the same time.

**Key point:** changing an ORM model does not automatically change the production database. The migration must be reviewed and applied.

# 9. PostgreSQL versus SQLite

Both are relational databases, but their architectures and operational characteristics differ.

| Aspect | SQLite | PostgreSQL |
|---|---|---|
| Architecture | Embedded database stored in a file | Client/server database |
| Setup | Very simple | Requires a running server or managed service |
| Concurrency | Multiple readers; writes have locking constraints | Designed for concurrent client workloads |
| Features | Compact feature set suitable for many embedded uses | Broad SQL features, indexing, constraints, and operational tooling |
| Common uses | Local development, prototypes, embedded apps, suitable small workloads | Production web services and multi-user applications |
| Connection management | Does not use a traditional remote client/server pool in the same way | Connection pooling is commonly important |

SQLite is not inherently unsuitable for production. It can work well when the workload, concurrency pattern, and deployment architecture fit its characteristics.

PostgreSQL is often a strong choice for a multi-user web application with concurrent writes and operational requirements.

Do not assume an application that works with SQLite will behave identically with PostgreSQL. Differences in locking, data types, SQL behavior, constraints, and supported features can surface during deployment.

# 10. Connection Pooling

Opening a new database connection for every query can be expensive. A connection pool keeps reusable connections available to the application.

Typical lifecycle:

`Request → acquire connection → execute SQL → release connection to pool`

A SQLAlchemy session is not itself a connection. The session manages ORM state and transactions; the engine and pool manage access to database connections.

## Common pool settings

- **Pool size:** number of connections maintained in the pool.
- **Max overflow:** additional temporary connections allowed beyond the base pool size.
- **Pool timeout:** how long a request waits to obtain a connection before failing.
- **Recycle:** a configured connection lifetime after which a connection may be replaced when checked out.

The exact behavior depends on the engine and pool configuration.

## What happens when the pool is exhausted?

If all available connections are checked out, another request needing a connection must wait. If a connection does not become available within the configured timeout, the request may fail with a pool timeout.

Possible causes:
- Sessions are not being closed correctly.
- Transactions remain open too long.
- Queries are slow.
- Too much concurrent work is directed at the database.
- The pool and database capacity are poorly matched.

## Troubleshooting sequence

1. Check session lifecycle and cleanup.
2. Identify long-running transactions and queries.
3. Inspect database load and lock waits.
4. Review application concurrency.
5. Tune pool settings using observed workload and database limits.

Simply increasing the pool size can make the problem worse if the database is already overloaded.

In [ ]:
# Illustrative synchronous SQLAlchemy session dependency for FastAPI.
# `engine` must be configured for the target database.

from collections.abc import Generator
from sqlalchemy.orm import Session, sessionmaker

SessionLocal = sessionmaker(bind=engine)

def get_db() -> Generator[Session, None, None]:
    db = SessionLocal()
    try:
        yield db
    finally:
        db.close()

The `finally` block closes the session even if the endpoint raises an exception. Closing a session releases its checked-out connection when appropriate; it does not necessarily close the underlying physical connection, because that connection may return to the pool for reuse.

Transaction boundaries should be deliberate. Some applications commit inside service functions; others use a unit-of-work pattern. Whichever design is chosen, it should be clear who owns commit and rollback behavior.

# 11. Preventing Overselling: Transactions and Concurrency

Consider a product with one unit left in stock. Two requests arrive at almost the same time. Both read `stock = 1`, and both attempt to create an order.

A naive sequence—read stock, then decrement it—can allow both requests to believe the item is available. This is a race condition.

Possible solutions include:
- An atomic conditional update.
- Row-level locking, where supported and appropriate.
- Suitable transaction isolation and retry behavior.
- Database constraints that help preserve invariants.

## Atomic conditional update

A useful pattern is to decrement stock only if sufficient stock remains:

```sql
UPDATE products
SET stock = stock - 1
WHERE id = :product_id
  AND stock >= 1;
```

The application checks the number of affected rows:
- One row updated: stock was available and the decrement succeeded.
- Zero rows updated: the product was missing or stock was insufficient.

The order and stock changes must still be coordinated in a transaction. If the transaction cannot complete, roll it back.

This pattern avoids relying solely on a separate read followed by an unprotected write. The exact implementation should account for database behavior and the application's transaction design.

# 12. End-to-End: Creating an Order

A request to create an order brings the concepts together.

1. **Request validation:** validate product IDs, quantities, and request shape.
2. **Authentication:** identify the caller.
3. **Authorization:** ensure the caller can place the order and access the relevant resources.
4. **Session acquisition:** obtain a database session through dependency injection.
5. **Transaction:** check availability safely, create the order and order items, and update inventory.
6. **Error handling:** roll back if a required operation fails; translate expected failures into appropriate HTTP responses.
7. **Commit:** commit the completed database operation.
8. **Response:** serialize the result through a response schema, typically using HTTP `201 Created`.
9. **Cleanup:** close the session so resources can be reused.
10. **Schema evolution:** use reviewed Alembic migrations when the database structure changes.

The exact ordering of checks and writes depends on the concurrency strategy. Inventory checks must be protected against concurrent requests, not merely performed before the transaction.

# 13. Interview Review: Questions and Model Answers

## 1. What is the difference between `add`, `flush`, `commit`, and `refresh`?

`add` places an object under session tracking. `flush` sends pending SQL operations to the database within the transaction. `commit` commits the transaction. `refresh` reloads an object's values from the database. A failed transaction may require rollback before the session can be reused.

## 2. What is the difference between a foreign key and an ORM relationship?

A foreign key is a database constraint that references another table. An ORM relationship allows application code to navigate between related objects. The relationship does not replace the foreign-key constraint.

## 3. What is the N+1 query problem?

It occurs when loading a collection triggers additional queries for related objects, often one query per item. Inspect the SQL and choose an appropriate loading strategy such as `selectinload` or `joinedload`.

## 4. Why is Alembic used?

Alembic manages versioned database schema changes. It helps evolve a database without recreating tables and losing data. Generated migrations must be reviewed, tested, and applied through a controlled deployment process.

## 5. What happens if order creation fails halfway through?

Use a transaction for the related database operations. If a required operation fails before commit, roll back so partial changes do not remain. Concurrency-sensitive inventory updates need additional safeguards.

## 6. What is the difference between a session and a connection?

The session manages ORM state and transactions. A connection is the resource used to communicate with the database. The engine and its pool manage connection acquisition and reuse.

## 7. Why use connection pooling?

Pooling reuses connections and reduces the overhead of repeatedly establishing them. If the pool is exhausted, requests may wait or time out. Investigate session cleanup, long transactions, query performance, and capacity before increasing pool limits.

## 8. When would you choose PostgreSQL over SQLite?

PostgreSQL is often a better fit for a concurrent, multi-user web service with substantial write activity and server-side operational requirements. SQLite remains appropriate for many local, embedded, and smaller workloads. The choice depends on actual workload and deployment constraints.

## 9. How do you model an order with multiple products?

Use `Order`, `Product`, and `OrderItem` models. `OrderItem` references both the order and product and stores fields such as quantity and purchase-time unit price.

## 10. How do you prevent two customers from buying the last unit?

Avoid a separate, unprotected stock check followed by a decrement. Use an atomic conditional update or suitable locking within a transaction, check whether the update succeeded, and coordinate it with order creation.

# 14. Final Quick-Recall Sheet

- `add()` tracks an ORM object.
- `flush()` sends pending SQL without committing.
- `commit()` commits the transaction.
- `rollback()` recovers from a failed transaction.
- `refresh()` reloads values from the database.
- A foreign key enforces a reference; an ORM relationship provides object navigation.
- A many-to-many relationship often needs an association model when the link has its own attributes.
- N+1 means one collection query plus extra queries for related objects.
- `selectinload` and `joinedload` can reduce unnecessary round trips.
- Alembic manages versioned schema migrations; autogeneration needs review.
- PostgreSQL is a common choice for concurrent production workloads; SQLite suits workloads that fit its characteristics.
- A session is not a physical connection.
- Connection pools reuse database connections.
- Pool exhaustion can result from long queries, long transactions, poor cleanup, or excess concurrency.
- Prevent race conditions with appropriate transactional and atomic database operations.
- Response schemas define output shape but do not replace authorization.

# 15. Suggested Revision Sequence

For a short preparation window, revise in this order:

1. **Transactions:** commit, flush, rollback, and atomicity.
2. **Relationships:** foreign keys, one-to-many, many-to-many, and cascade choices.
3. **Performance:** N+1 queries and eager loading.
4. **Migrations:** Alembic revisions, autogeneration, upgrade/downgrade, and safe rollout.
5. **Pooling:** session lifecycle, connection reuse, exhaustion, and troubleshooting.
6. **Integration:** explain a complete order-creation request from validation to response.

For each topic, be ready to explain:
- What it is.
- Why it is needed.
- How it works in the application.
- One common failure mode.
- One trade-off or design decision.